In [1]:
import requests
from wikidata.client import Client

In [9]:

def wd_by_an_identifier(property, idstring):
    """
    Query Wikidata for an entity with the given TLG author ID (property P1266).
    Returns JSON results from the WDQS endpoint.
    """
    query = f"""
    SELECT ?item ?itemLabel WHERE {{
      ?item wdt:{property} "{idstring}" .
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    """

    url = "https://query.wikidata.org/sparql"
    headers = {"Accept": "application/sparql-results+json"}

    response = requests.get(url, params={"query": query, "format": "json"}, headers=headers)
    response.raise_for_status()

    return response.json()

In [10]:
wd_by_an_identifier("P3576", "0086")

{'head': {'vars': ['item', 'itemLabel']},
 'results': {'bindings': [{'item': {'type': 'uri',
     'value': 'http://www.wikidata.org/entity/Q868'},
    'itemLabel': {'xml:lang': 'en',
     'type': 'literal',
     'value': 'Aristotle'}}]}}

In [11]:
wd_by_an_identifier("P214", "78769600")

{'head': {'vars': ['item', 'itemLabel']},
 'results': {'bindings': [{'item': {'type': 'uri',
     'value': 'http://www.wikidata.org/entity/Q1541'},
    'itemLabel': {'xml:lang': 'en', 'type': 'literal', 'value': 'Cicero'}}]}}

In [17]:
headers = {
        "User-Agent": "MyResearchBot/0.1 (kase@ff.zcu.cz)"
    }
def get_entity_json(qid, headers = headers):
    url = f"https://www.wikidata.org/wiki/Special:EntityData/{qid}.json"
    resp = requests.get(url, headers=headers)
    resp.raise_for_status()
    return resp.json()

In [15]:
data = get_entity_json("Q868")  # Aristotle
print(data.keys())

dict_keys(['entities'])


In [16]:
data["entities"]

{'Q868': {'pageid': 1188,
  'ns': 0,
  'title': 'Q868',
  'lastrevid': 2424924785,
  'modified': '2025-11-03T08:43:55Z',
  'type': 'item',
  'id': 'Q868',
  'labels': {'en': {'language': 'en', 'value': 'Aristotle'},
   'es': {'language': 'es', 'value': 'Aristóteles'},
   'fr': {'language': 'fr', 'value': 'Aristote'},
   'de': {'language': 'de', 'value': 'Aristoteles'},
   'it': {'language': 'it', 'value': 'Aristotele'},
   'ilo': {'language': 'ilo', 'value': 'Aristoteles'},
   'ko': {'language': 'ko', 'value': '아리스토텔레스'},
   'ru': {'language': 'ru', 'value': 'Аристотель'},
   'en-ca': {'language': 'en-ca', 'value': 'Aristotle'},
   'en-gb': {'language': 'en-gb', 'value': 'Aristotle'},
   'af': {'language': 'af', 'value': 'Aristoteles'},
   'gsw': {'language': 'gsw', 'value': 'Aristoteles'},
   'am': {'language': 'am', 'value': 'አሪስጣጣሊስ'},
   'an': {'language': 'an', 'value': 'Aristótil'},
   'ang': {'language': 'ang', 'value': 'Aristoteles'},
   'ar': {'language': 'ar', 'value': 'أرسطو

In [18]:
def get_claim_value(qid, pid, headers=headers):
    url = "https://www.wikidata.org/w/api.php"
    params = {
        "action": "wbgetentities",
        "ids": qid,
        "props": "claims",
        "format": "json"
    }
    r = requests.get(url, params=params, headers=headers)
    r.raise_for_status()
    data = r.json()

    claims = data["entities"][qid]["claims"]

    if pid not in claims:
        return None  # property not present

    # take the first value
    mainsnak = claims[pid][0]["mainsnak"]
    datavalue = mainsnak.get("datavalue", {})
    return datavalue.get("value")

In [20]:
get_claim_value("Q9047", "P7935")

'Leibniz_Gottfried_Wilhelm'